# 300 m walkability computation

**Goal:** Identify which residential buildings (street number to be precise for each inhabitant in the city) in a given city are within a **300m walking distance** of a publicly accessible park or green space.

**Overview:**
1. Load: boundary, civic number, greenery, buildings polygons and pedestrian network
2. Detect park entrance points 
3. Generate 300m walking isochrones (from entrance) - GraphHopper
4. Building - entrance walk distances (for matched buildings only)
5. Results (`{city}_walkability_map.gpkg`)

---

> **GraphHopper must be running before executing the distance cells.**  
> In a separate terminal, navigate to the folder containing `graphhopper-web-11.0.jar`, `config.yml`, and `{city}.osm.pbf`, then run:
> ```bash
> rm -rf graph-cache
> java -jar graphhopper-web-11.0.jar server config.yml
> ```
> Wait for `Started server at HTTP 8989` before continuing.
> The port is assigned automatically from config — see `get_gh_port(CITY)`.

In [1]:
# ── CHANGE THIS TO SWITCH CITY ──────────────────────────────────────────────
CITY = "milano"


In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import MultiPoint, Point, shape
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from tqdm import tqdm

os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

from config import CRS_WGS, CRS_METRIC, WALKING_DISTANCE, EUCLID_CUTOFF, get_city_cfg, get_gh_port, get_output_paths

cfg   = get_city_cfg(CITY)
src   = cfg["data_sources"]
paths = get_output_paths(CITY)

GH_PORT = get_gh_port(CITY)
ISO_URL = f"http://localhost:{GH_PORT}/isochrone"
GH_URL  = f"http://localhost:{GH_PORT}/route"

print(f"City   : {CITY}")
print(f"GH port: {GH_PORT}")
print(f"Output : {paths['gpkg']}")

City   : milano
GH port: 8990
Output : /Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index/data/300m/milano/milano_walkability_map.gpkg


## 1 - Load Input Data

| Layer | Source | info |
|---|---|---|
| Municipal boundary | `{city}_boundary.gpkg` | area of interest |
| Buildings with civic numbers | `{city}_buildings_streetnum.gpkg` | street numbers |
| Green spaces | `{city}_greenery.gpkg` | output of greenery_data_exploration.ipynb |

+ OpenStreetMap pedestrian network

In [3]:
# ── Boundary ─────────────────────────────────────────────────────────────────
boundary = gpd.read_file(src["boundary"]).to_crs(CRS_METRIC)
boundary_wgs84 = boundary.to_crs(CRS_WGS).geometry.union_all()
print(f"Boundary loaded: {CITY}")

Boundary loaded: milano


In [5]:
# ── Civic numbers (address points) ───────────────────────────────────────────
civici = (
    gpd.read_file(src["buildings_sn"])
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.notna()]
    .reset_index(drop=True)
)
civici["building_id"] = civici[cfg["building_id_col"]].astype(str)
print(f"{len(civici):,} civici loaded")

64,973 civici loaded


In [6]:
# ── Green spaces ─────────────────────────────────────────────────────────────
# Loads all park layers listed in cfg["park_layers"].
# Each .gpkg is the output of greenery_data_exploration.ipynb and contains
# layers: osm_public  and/or  municipality  (skip whichever doesn't exist).

import fiona

GREENERY_LAYERS = ["osm_public", "municipality"]   # standard layer names from exploration nb

park_gdfs = []
for park_key in cfg["park_layers"]:
    fpath = src[park_key]
    available = fiona.listlayers(fpath)
    for layer in GREENERY_LAYERS:
        if layer in available:
            gdf = gpd.read_file(fpath, layer=layer)
            park_gdfs.append(gdf)
            print(f"  loaded {len(gdf):,} features — {park_key}/{layer}")

parks = (
    pd.concat(park_gdfs, ignore_index=True)
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
    .assign(park_id=lambda df: df.index)
)

print(f"\n{len(parks):,} parks total")
print(parks["layer_type"].value_counts())

  loaded 1,557 features — parks_municipal/osm_public
  loaded 1,557 features — parks_municipal/municipality

3,114 parks total
layer_type
osm_public      1557
municipality    1557
Name: count, dtype: int64


In [7]:
from shapely.strtree import STRtree

def deduplicate_parks(parks: gpd.GeoDataFrame, overlap_threshold: float = 0.7) -> gpd.GeoDataFrame:
    """
    Remove OSM polygons that are substantially covered by a municipality polygon.
    Logic: if an OSM park overlaps > overlap_threshold of its own area with any
    municipality park, drop it — the municipality version is more authoritative.
    """
    muni = parks[parks["layer_type"] == "municipality"].copy()
    osm  = parks[parks["layer_type"] == "osm_public"].copy()

    if muni.empty or osm.empty:
        return parks

    muni_tree = STRtree(muni.geometry.values)

    drop_idx = []
    for idx, row in osm.iterrows():
        candidates = muni_tree.query(row.geometry, predicate="intersects")
        if len(candidates) == 0:
            continue
        overlap = sum(
            row.geometry.intersection(muni.geometry.iloc[c]).area
            for c in candidates
        )
        if overlap / row.geometry.area >= overlap_threshold:
            drop_idx.append(idx)

    print(f"Dropping {len(drop_idx)} OSM parks covered by municipality polygons")
    return parks.drop(index=drop_idx).reset_index(drop=True)


parks = (
    pd.concat(park_gdfs, ignore_index=True)
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
)
parks = deduplicate_parks(parks)
parks = parks.assign(park_id=lambda df: df.index)   # reassign after dropping

print(f"\n{len(parks):,} parks after deduplication")
print(parks["layer_type"].value_counts())

Dropping 1557 OSM parks covered by municipality polygons

1,557 parks after deduplication
layer_type
municipality    1557
Name: count, dtype: int64


## 1. Pedestrian network 
Download the full walk network for Bologna from OSM and project to UTM 32N.


In [8]:
G = ox.graph_from_polygon(boundary_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)

_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"{len(edges):,} pedestrian edges")

256,372 pedestrian edges


## 2. Entrance 

In [9]:
OSM_SNAP_TOL = 2.0   # m => OSM node must be within 2 m from the boundary
DEDUP_DIST   = 5.0   # m => cluster intersection points closer than this
SNAP_RADIUS  = 150   # m — base centroid-snap search radius
 
 
def extract_points(geom) -> list:
    """Extract all Point objects from any Shapely geometry type."""
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t == "Point":
        return [geom]
    if t == "MultiPoint":
        return list(geom.geoms)
    if t == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []
 
 
def cluster_points(pts: list, dist: float) -> list:
    """
    Greedy single-linkage: merge points within `dist` m into one centroid.
    Prevents one path crossing from producing 3–4 near-identical entrance pts.
    """
    if not pts:
        return []
    remaining = list(pts)
    clusters  = []
    while remaining:
        seed  = remaining.pop(0)
        group = [seed]
        far   = []
        for p in remaining:
            (group if seed.distance(p) <= dist else far).append(p)
        remaining = far
        clusters.append(MultiPoint(group).centroid)
    return clusters
 

# 2a. Entrance - OSM

In [94]:
osm_entrances_raw = ox.features_from_polygon(
    boundary_wgs84,
    tags={"entrance": ["yes", "main", "secondary", "service",
                       "staircase", "emergency", "exit"]}
).to_crs(CRS_METRIC)
 
osm_entrance_pts = osm_entrances_raw[
    osm_entrances_raw.geometry.geom_type == "Point"
].copy()
print(f"OSM entrance nodes found: {len(osm_entrance_pts)}")
 

OSM entrance nodes found: 14592


## 2b. Entrance - DETECTION

1. Explicit OSM entrance=* node within 2 m of park boundary
2. Intersection of park boundary with pedestrian edges
3. Park centroid to nearest edge (OSM parks only)

In [ ]:
# Pre-compute edge midpoints once (used in every park iteration)
edges_geom = edges.geometry.values
edge_mids  = np.array([
    [g.interpolate(0.5, normalized=True).x,
     g.interpolate(0.5, normalized=True).y]
    for g in edges_geom
])

osm_entrance_tree = STRtree(osm_entrance_pts.geometry.values)
edge_tree  = STRtree(edges_geom)

rows= []
step_counts = {1: 0, 2: 0, 3: 0, "3_extended": 0}

for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    geom     = park.geometry
    boundary = geom.boundary
    park_id  = park["park_id"]
    pts      = []

    #Edges: external => midpoint outside the park polygon.
    #Buffer by 1 m so edges that just touch the boundary are included.

    cand_idxs     = edge_tree.query(geom.buffer(1), predicate="intersects")
    external_idxs = [
        i for i in cand_idxs
        if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
    ]
    crossing_idxs = [
        i for i in external_idxs
        if boundary.intersects(edges_geom[i])
    ]
    #ENTRANCE COMPUTATION
    #1:explicit (OSM)
    osm_hits  = osm_entrance_tree.query(geom.buffer(OSM_SNAP_TOL),
                                        predicate="intersects")
    tier1_pts = [
        osm_entrance_pts.geometry.iloc[i]
        for i in osm_hits
        if boundary.distance(osm_entrance_pts.geometry.iloc[i]) <= OSM_SNAP_TOL
    ]
    if tier1_pts:
        pts = tier1_pts
        step_counts[1] += 1

    #2: boundary ∩ external edges + near-miss projection 
    if not pts:
        tier2_raw = []
        for idx in edge_tree.query(boundary, predicate="intersects"):
            # Skip edges fully contained inside the park (internal park paths)
            if not geom.contains(edges_geom[idx]):
                inter = boundary.intersection(edges_geom[idx])
                tier2_raw.extend(extract_points(inter))

        # Near-miss: edges within 5 m but not crossing
        #A park with no fence often has its boundary drawn a few metres inside the actual edge
        #or the street is mapped slightly offset.

        near_miss_idxs = [
            idx for idx in edge_tree.query(boundary.buffer(5), predicate="intersects")
            if not geom.contains(edges_geom[idx])
            and not boundary.intersects(edges_geom[idx])
        ]
        for idx in near_miss_idxs:
            proj_pt, _ = nearest_points(edges_geom[idx], boundary)
            tier2_raw.append(proj_pt)

        if tier2_raw:
            pts = cluster_points(tier2_raw, DEDUP_DIST)
            step_counts[2] += 1

    #3: centroid => nearest external edge (OSM parks only)
    if not pts and park["source"] == "OSM":
        boundary_search = list(edge_tree.query(
            geom.boundary.buffer(SNAP_RADIUS), predicate="intersects"
        ))
        ext_nearby = [
            i for i in boundary_search
            if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
        ]

        if ext_nearby:
            closest    = min(ext_nearby,
                             key=lambda i: edges_geom[i].distance(geom.boundary))
            snap_pt, _ = nearest_points(edges_geom[closest], geom.boundary)
            pts.append(snap_pt)
            step_counts[3] += 1

        #Named parks with no walkable edges nearby = expand search radius
        elif pd.notna(park.get("name")) and park.get("name"):
            far_search = list(edge_tree.query(
                geom.boundary.buffer(500), predicate="intersects"
            ))
            ext_far = [
                i for i in far_search
                if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
            ]
            if ext_far:
                closest    = min(ext_far,
                                 key=lambda i: edges_geom[i].distance(geom.boundary))
                snap_pt, _ = nearest_points(edges_geom[closest], geom.boundary)
                pts.append(snap_pt)
                step_counts["3_extended"] += 1

    for pt in pts:
        rows.append({"park_id": park_id, "geometry": pt})

entrances = gpd.GeoDataFrame(rows, crs=CRS_METRIC)
missing   = set(parks["park_id"]) - set(entrances["park_id"])

print(f"\n{len(entrances):,} entrance points across "
      f"{entrances['park_id'].nunique()} parks")

print(f"OSM: {step_counts[1]} | "
      f"boundary∩street: {step_counts[2]} | "
      f"centroid: {step_counts[3]} | "
      f"extended snap: {step_counts['3_extended']}")

print(f"Parks with no entrance: {len(missing)}")
if missing:
    print("  Missing IDs:", sorted(missing))

Park entrances: 100%|██████████| 2622/2622 [01:37<00:00, 26.85it/s]



12,011 entrance points across 2471 parks
Tiers  →  OSM nodes: 102 | boundary∩street: 2211 | centroid snap: 156 | extended snap: 2
Parks with no entrance: 151
  Missing IDs: [2, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 100, 107, 170, 172, 196, 204, 205, 207, 208, 209, 211, 220, 225, 244, 255, 257, 270, 277, 296, 301, 305, 309, 311, 314, 315, 318, 321, 322, 326, 327, 328, 332, 333, 348, 356, 371, 372, 373, 379, 383, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 443, 450, 451, 453, 455, 456, 498, 511, 514, 515, 519, 520, 525, 528, 553, 554, 567, 603, 654, 688, 704, 706, 708, 709, 710, 711, 713, 775, 776, 777, 778, 783, 785, 793, 817, 823, 829, 833, 854, 855, 857, 858, 860, 926, 930, 960, 962, 963, 965, 973, 976, 985, 989, 990, 992, 993, 994, 995, 996, 997, 1001, 1002, 1003, 1006, 1010, 1012, 1014, 1016, 1064, 1655, 1656, 1657, 1660, 1661, 1672, 1673, 1674, 1675, 1676, 1677, 1680, 1681, 1776, 1780, 2039, 2510, 2511, 2512, 2513, 2588]


## 2c. Civic numbers inside parks

In [ ]:
#force civici that lie inside a park 
civici_in_park = gpd.sjoin(
    civici[["building_id", "geometry"]],
    parks[["park_id", "geometry"]],
    how="inner",
    predicate="within",
).drop_duplicates("building_id")
 
civici_in_park_ids    = set(civici_in_park["building_id"])
civici_in_park_lookup = civici_in_park.set_index("building_id")["park_id"].to_dict()
 
print(f"Civici inside a park polygon: {len(civici_in_park_ids):,} "
      f"will be force-flagged within_300m=True")
 

Civici inside a park polygon: 321 will be force-flagged within_300m=True


## 3. Walking Isochrones (GraphHopper)
Each entrance => request a **300 m walking isochrone**
The isochrone is a polygon covering all locations reachable within 300 m of walking from that entrance.



In [102]:
def get_isochrone(lat, lon, distance_m=300):
    """
    Request a distance-based walking isochrone from GraphHopper.
    Returns the GeoJSON polygon feature, or None on failure.
    """
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None


# Convert entrances to WGS84 for GraphHopper
entrances_wgs = entrances.to_crs(CRS_WGS)
entrances["entrance_lat"] = entrances_wgs.geometry.y
entrances["entrance_lon"] = entrances_wgs.geometry.x

iso_rows = []
for _, row in tqdm(entrances.iterrows(), total=len(entrances), desc="Isochrones"):
    result = get_isochrone(row["entrance_lat"], row["entrance_lon"])
    if result:
        iso_rows.append({
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "geometry":     shape(result["geometry"]),
        })
 
# Isochrones stay in WGS84 from GH, project to metric for spatial join
isochrones = gpd.GeoDataFrame(
    iso_rows,
    geometry="geometry",   # ← explicit, never relies on column-name inference
    crs=CRS_WGS,
).to_crs(CRS_METRIC)

print(f"{len(isochrones):,} isochrones computed "
      f"({len(entrances) - len(isochrones):,} failed)")


Isochrones: 100%|██████████| 12011/12011 [59:11<00:00,  3.38it/s] 


11,944 isochrones computed (67 failed)


## 4. Spatial Join: Buildings Within 300 m Walk

Each civico that falls inside multiple overlapping isochrones will produce
- multiple rows — one per (civico, entrance) pair.
- this is intentional: Step 5 routes all pairs and keeps the shortest.

In [ ]:
civici_gdf = gpd.GeoDataFrame(
    {"building_id": civici["building_id"]},
    geometry=civici.geometry,   # already Point geometries in CRS_METRIC
    crs=CRS_METRIC,
)
 
joined = gpd.sjoin(
    civici_gdf,
    isochrones[["park_id", "entrance_lat", "entrance_lon", "geometry"]],
    how="left",
    predicate="within",
)
 
# All valid (civico, entrance) candidate pairs
candidates = (
    joined[joined["park_id"].notna()]
    .copy()
    .reset_index(drop=True)
)
 
# WGS84 coordinates of each civico for GraphHopper routing
cands_wgs = gpd.GeoDataFrame(
    candidates[["building_id"]],
    geometry=candidates.geometry,
    crs=CRS_METRIC,
).to_crs(CRS_WGS)

candidates["orig_lat"] = cands_wgs.geometry.y
candidates["orig_lon"] = cands_wgs.geometry.x
 
n_pairs = len(candidates)
n_cands = candidates["building_id"].nunique()
print(f"{n_cands:,} candidate civici ({n_cands / len(civici) * 100:.1f}% of total)")
print(f"{n_pairs:,} (civico, entrance) pairs to route")
print(f"avg {n_pairs / max(n_cands, 1):.1f} entrance candidates per civico")
 


3,488 candidate civici (5.4% of total)
774,282 (civico, entrance) pairs to route
avg 222.0 entrance candidates per civico


## 5. Route all pairs → keep shortest walk per civico 
- GraphHopper gives the true walking distance along the street network.
- Routing to every candidate entrance (not just one) ensures we find the
- genuinely nearest entrance by walking, not by straight-line distance.

In [ ]:
#point-to-point routing call 

def graphhopper(orig_lat, orig_lon, dest_lat, dest_lon):

    params = {
        "point":       [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":     "foot",
        "calc_points": "false",   # geometry not needed — distance only
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

In [ ]:
results = []

for _, row in tqdm(candidates.iterrows(), total=n_pairs,
                   desc="Routing civico to entrance"):
    dist = graphhopper(
        row["orig_lat"],     row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"],
    )
    if dist is not None:
        results.append({
            "building_id":  row["building_id"],
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "walk_dist_m":  dist,
        })
 
results_df = pd.DataFrame(results)
print(f"{len(results_df):,} pairs routed ({n_pairs - len(results_df):,} GH failures)")
 
# Per civico: keep the entrance with the shortest WALKING distance
best = (
    results_df
    .sort_values("walk_dist_m")
    .groupby("building_id", sort=False)
    .first()
    .reset_index()
)

Routing civico to entrance:  76%|███████▋  | 591385/774282 [2:10:34<55:03, 55.37it/s]    

## 6. final output 

In [ ]:
out = civici.merge(
    best[["building_id", "park_id", "entrance_lat", "entrance_lon", "walk_dist_m"]],
    on="building_id",
    how="left",
)

out["is_candidate"] = out["building_id"].isin(candidates["building_id"])
out["within_300m"]  = out["walk_dist_m"].le(WALKING_DISTANCE)

# Civici physically inside a park polygon → force True
in_park_mask = out["building_id"].isin(civici_in_park_ids)
out.loc[in_park_mask, "within_300m"]  = True
out.loc[in_park_mask, "walk_dist_m"]  = out.loc[in_park_mask, "walk_dist_m"].fillna(0.0)
out.loc[in_park_mask, "park_id"]      = (
    out.loc[in_park_mask, "building_id"]
    .map(civici_in_park_lookup)
    .combine_first(out.loc[in_park_mask, "park_id"])
)

# Rename to clean output columns
# Extend this dict if your source file uses different column names
rename = {
    cfg["building_id_col"]:  "street_number",   # source address ID → canonical name
    "indirizzo_completo":    "address",
    "park_id":               "nearest_park_id",
    "layer_type":            "park_type",
    "area_ha":               "park_area_ha",
    "source":                "park_source",
}
# neighbourhood column varies by city
if cfg["neighbourhood_col"] and cfg["neighbourhood_col"] in out.columns:
    rename[cfg["neighbourhood_col"]] = "neighbourhood"

if "name" in out.columns:
    rename["name"] = "park_name"

out = out.rename(columns=rename)
out = gpd.GeoDataFrame(out, geometry="geometry", crs=CRS_METRIC)

n_total   = len(out)
n_within  = int(out["within_300m"].sum())
n_in_park = int(in_park_mask.sum())
n_routed  = int((out["is_candidate"] & out["walk_dist_m"].notna()).sum())
n_no_gh   = int((out["is_candidate"] & out["walk_dist_m"].isna()).sum())
n_no_iso  = int((~out["is_candidate"]).sum())

print(f"\n300m PARK ACCESS — {CITY.upper()}")
print(f"Total civici:                    {n_total:,}")
print(f"Within 300 m walk:               {n_within:,}  ({n_within / n_total * 100:.1f}%)")
print(f"  of which routed by GH:         {n_routed:,}")
print(f"  of which inside park (forced): {n_in_park:,}")
print(f"Candidate, no GH route found:    {n_no_gh:,}")
print(f"No isochrone match at all:       {n_no_iso:,}")

Total civici:                    77,630
Within 300 m walk:               59,266  (76.3%)
  of which routed by GraphHopper:61,360
  of which inside park (forced): 615
Candidate, no GH route found:    0
No isochrone match at all:       16,270


## 7. Results

In [ ]:
# GPKG (buildings + parks)
out.to_file(paths["gpkg"], driver="GPKG", layer="buildings")
parks[parks["park_id"].isin(out.get("nearest_park_id", []))].to_file(
    paths["gpkg"], driver="GPKG", layer="parks"
)

# GeoJSON + CSV (WGS84)
out_wgs = out.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y

final_cols = [
    "street_number", "address", "longitude", "latitude",
    "neighbourhood", "nearest_park_id", "park_type", "park_source",
    "park_area_ha", "entrance_lat", "entrance_lon",
    "walk_dist_m", "within_300m", "geometry",
]
if "park_name" in out_wgs.columns:
    final_cols.insert(final_cols.index("park_type"), "park_name")

out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]

out_wgs.to_file(paths["geojson"], driver="GeoJSON")
out_wgs.drop(columns="geometry").to_csv(paths["csv"], index=False)

print(f"Written:")
print(f"  {paths['gpkg']}")
print(f"  {paths['geojson']}")
print(f"  {paths['csv']}")

data/300m/300_walkability_map.gpkg  (buildings + parks layers)
data/300m/buildings_300m_indicator.geojson
data/300m/buildings_300m_indicator.csv
  street_number                             address  longitude   latitude          neighbourhood  nearest_park_id  entrance_lat  entrance_lon  walk_dist_m  within_300m                   geometry
0          27/1        VIA MARCO EMILIO LEPIDO 27/1  11.278541  44.516819  Borgo Panigale - Reno            921.0     44.517364     11.278157       79.001         True  POINT (11.27854 44.51682)
1          46/2  VIA GIOVANNI BATTISTA GANDINO 46/2  11.355349  44.478504          Santo Stefano            216.0     44.479283     11.354519      111.623         True   POINT (11.35535 44.4785)
2           9/2              VIA GUIDO ZUCCHINI 9/2  11.351607  44.501726          Santo Stefano           1397.0     44.501100     11.351751       98.923         True  POINT (11.35161 44.50173)
